# Historical archive: NewFewShotsGemma2

**Status: not independently reproduced.**

Checkpoint / purpose: `google/gemma-2b-it`.

Historical code, not independently reproduced. Saved settings can differ from report labels; notebook filenames are not authoritative configurations. Exploratory classification and prompting variants. The report describes NaN failures; no verified comparable performance claim is made.

Publication edits only: credentials use `HF_TOKEN`; personal absolute paths were relocated; saved outputs, attachments and runtime metadata were removed. Historical installation cells and local data dependencies are retained. Review code before running; it can download models, install packages and start expensive training. See `docs/experiment_catalogue.md` and the new pipeline for supported experiments.


In [ ]:
from IPython.display import clear_output
clear_output()

## Fine-Tunining Gemma-2-2b-it on ECTHR_A using few-shots technique and LoRA

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import Dataset, load_dataset
import pandas as pd
import os
import numpy as np
from peft import TaskType, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding)

import bitsandbytes as bnb

import torch
print(torch.cuda.is_available())

### Defining Article-id, id-Article Mapping

In [ ]:
id_mapping = {
    0: 'Article 1: Obligation to respect human rights',
    1: 'Article 2: Right to life',
    2: 'Article 3: Prohibition of torture',
    3: 'Article 4: Prohibition of slavery and forced labour',
    4: 'Article 5: Right to liberty and security',
    5: 'Article 6: Right to a fair trial',
    6: 'Article 7: No punishment without law',
    7: 'Article 8: Right to respect for private and family life',
    8: 'Article 9: Freedom of thought, conscience and religion',
    9: 'Article 10: Right to freedom of speech and expression',
    10: 'No Articles Violated'
}

article_mapping = {
    'Article 1: Obligation to respect human rights': 0,
    'Article 2: Right to life': 1,
    'Article 3: Prohibition of torture': 2,
    'Article 4: Prohibition of slavery and forced labour': 3,
    'Article 5: Right to liberty and security': 4,
    'Article 6: Right to a fair trial': 5,
    'Article 7: No punishment without law': 6,
    'Article 8: Right to respect for private and family life': 7,
    'Article 9: Freedom of thought, conscience and religion': 8,
    'Article 10: Right to freedom of speech and expression': 9,
    'No Articles Violated': 10
}

### Compute Metrics Function

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Convert logits to binary predictions using a threshold (e.g., 0.5)
    predictions = (logits > 0.5).astype(np.float32)
    
    # Calculate accuracy (for multi-label, you may want to use a custom definition of accuracy)
    accuracy = accuracy_score(labels, predictions)
    
    # Calculate micro and macro precision, recall, and F1 scores
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, predictions, average='micro')
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    
    # Return a dictionary of all the metrics
    return {
        "accuracy": accuracy,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro
    }


### Step 1: Model Loading and Configuration

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for potentially higher accuracy (optional)
    bnb_4bit_quant_type="nf4",  # Quantization type (specifics depend on hardware and library)
)

In [ ]:
model_id = "google/gemma-2b-it"
access_token = __import__('os').environ['HF_TOKEN']  

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,  
    num_labels=11,  # Number of output labels
    problem_type="multi_label_classification", 
    id2label=id_mapping,  
    label2id=article_mapping,  
    quantization_config=bnb_config,  # configuration for quantization 
    token = access_token
)

In [ ]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=64,  # Reduction factor (lower r means more parameters in the adapter)
    lora_alpha=32,  # Dimensionality of the adapter projection
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"],  # List of modules to apply the LoRA adapter
    lora_dropout=0.05,  # Dropout rate for the adapter
    bias="none",  # Bias configuration for the adapter
    task_type="SEQ_CLS"  # Task type (sequence classification in this case)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
print(f' Vocab size of the model {model_id}: {len(tokenizer.get_vocab())}')

In [ ]:
early_stop = EarlyStoppingCallback(early_stopping_patience=1, early_stopping_threshold=.0)

### Step 2:  Data Processing

In [ ]:
train = pd.read_parquet('ecthr_a_article_desc_train.parquet')
test = pd.read_parquet('ecthr_a_article_desc_test.parquet')
validation = pd.read_parquet('ecthr_a_article_desc_validation.parquet')


In [ ]:
from datasets import Dataset
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

def preprocess_function(train, test, validation, tokenizer, instructions, subset_ratio=0.05, seed=42):
    # Function to map articles to their corresponding IDs
    def map_articles(articles):
        return [article_mapping[article] for article in articles]
    
    # Apply the mapping function to each dataset
    train['labels'] = train['articles'].apply(map_articles)
    test['labels'] = test['articles'].apply(map_articles)
    validation['labels'] = validation['articles'].apply(map_articles)

    # Convert labels to a multi-hot encoded format using MultiLabelBinarizer
    mlb = MultiLabelBinarizer(classes=range(11))

    train_labels = mlb.fit_transform(train['labels'])
    test_labels = mlb.transform(test['labels'])
    validation_labels = mlb.transform(validation['labels'])

    # Convert the dataframes to Hugging Face datasets
    train_dataset = Dataset.from_pandas(train[['text']])
    test_dataset = Dataset.from_pandas(test[['text']])
    validation_dataset = Dataset.from_pandas(validation[['text']])

    # Add the multi-hot encoded labels in float format
    train_dataset = train_dataset.add_column('labels', list(train_labels.astype(np.float32)))
    test_dataset = test_dataset.add_column('labels', list(test_labels.astype(np.float32)))
    validation_dataset = validation_dataset.add_column('labels', list(validation_labels.astype(np.float32)))

    # Select a subset of the training dataset
    train_subet = train_dataset.shuffle(seed=seed).select(range(int(len(train_dataset) * subset_ratio)))

    # Tokenize the datasets with instructions
    def tokenize_function(examples):
        inputs = [f"{instructions} {text}" for text in examples['text']]
        return tokenizer(inputs, max_length=4096, truncation=True, padding="max_length")

    # Apply tokenization
    tokenized_train_dataset = train_subet.map(tokenize_function, batched=True)
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
    tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)

    return tokenized_train_dataset, tokenized_test_dataset, tokenized_validation_dataset


In [ ]:
# Using preprocess function 
# 0.05% of train dataset
instructions = "Based on the following case, tell me which articles of the ECHR may have been violated: \n"
tokenized_train_dataset, tokenized_test_dataset, tokenized_validation_dataset = preprocess_function(train, test, validation, tokenizer, instructions)

In [ ]:
training_args = TrainingArguments(
    output_dir="step_weights",  
    learning_rate=1e-5,  
    per_device_train_batch_size=3,  # Batch size per device
    per_device_eval_batch_size=2,  # Batch size per device for evaluation 
    #num_train_epochs=1,  # Set to 1 or remove since we are using steps
    max_steps=50,  # Total number of training steps to perform
    logging_steps=2,
    gradient_accumulation_steps=2,  # Accumulate gradients over 2 forward passes
    weight_decay=0.01,  # Weight decay for regularization
    eval_strategy='steps',  # Change from 'epoch' to 'steps'
    eval_steps=50,  # Number of steps to perform an evaluation
    save_strategy="steps",  # Change from 'epoch' to 'steps'
    save_steps=50,  # Number of steps after which to save model checkpoints
    load_best_model_at_end=True,  # Load the best model based on the chosen metric
    optim="adamw_8bit",
    report_to="none",  # Disable logging to Weight&Bias
    metric_for_best_model='eval_loss',  # Metric to monitor for loading the best model
)

In [ ]:
trainer = Trainer(
    model=model,  # The LoRA-adapted model
    args=training_args,  # Training arguments
    train_dataset= tokenized_train_dataset,  # Training dataset 0.2%
    eval_dataset= tokenized_test_dataset,  # Evaluation dataset
    tokenizer=tokenizer,  # Tokenizer for processing text
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),  # Data collator for preparing batches
    compute_metrics=compute_metrics,  # Function to calculate evaluation metrics
    callbacks=[early_stop]  # Optional early stopping callback
)

In [ ]:
trainer.evaluate()

In [ ]:
trainer.train()

In [ ]:
!uname -r